# ⚙️ Der Optimizer ist dein Compiler

Traditionell: **Quellcode → Compiler → Binary**
Heute: **Signature + Metrik + Daten → Der Optimizer → Optimierter Prompt**

Beides nimmt menschenlesbare Spezifikationen und produziert maschinenausführbare Artefakte.

Das Werkzeug optimiert hier automatisch — du musst keinen einzigen Prompt von Hand schreiben!

In [ ]:
import sys; sys.path.insert(0, ".")
import dspy, ipywidgets as widgets
from IPython.display import display
from dspy_tasks.tasks import get_task
from dspy_tasks.actions import run_baseline, run_optimization
from dspy_tasks.visualize import *
from dspy_tasks.config import get_available_models, get_default_model, configure_dspy

MODELS = get_available_models()
model_dd = model_picker(MODELS, default=get_default_model())
opt_dd = optimizer_picker()
display(widgets.VBox([model_dd, opt_dd]))

In [ ]:
from dspy_tasks.visualize import diagram

diagram([
    {"label": "Signature", "detail": "Was du willst", "icon": "📝", "color": "#0078d4"},
    {"label": "Metrik", "detail": "Was 'gut' heisst", "icon": "📐", "color": "#0078d4"},
    {"label": "Trainingsdaten", "detail": "Beispiele", "icon": "📊", "color": "#0078d4"},
    {"label": "Optimizer", "detail": "probiert Varianten", "icon": "⚙️", "color": "#ca5010"},
    {"label": "Optimierter Prompt", "detail": "+ Few-Shot Demos", "icon": "🎯", "color": "#107c10"},
], title="Die Optimierungs-Pipeline")

## ✏️ Erst du, dann die Maschine

Bevor wir den automatischen Optimizer loslassen, versuch es erstmal selbst! Editiere den Prompt unten und schau, welchen Score du erreichst. Danach vergleichen wir mit dem automatisch optimierten Ergebnis.

In [ ]:
from dspy_tasks.visualize import prompt_workshop

# Dein manueller Versuch — editiere den Prompt!
manual_workshop = prompt_workshop(
    task_id="multihop_qa",
    default_instructions="Read the context carefully and answer the question. Think step by step.",
    max_eval=6,
)
display(manual_workshop)


## ⚙️ Und jetzt die Maschine...

Du hast deinen besten manuellen Score gesehen. Jetzt drück unten auf "Optimieren" und schau, was passiert. Der Optimizer probiert systematisch hunderte Varianten durch und findet die beste — in Sekunden.

**Die Frage ist:** Kann der Computer einen besseren Prompt finden als du?

## BootstrapFewShot: Der schnelle Compiler

**BootstrapFewShot** ist wie `-O1` Optimierung — schnell und effektiv. Er sucht die besten Few-Shot-Beispiele aus deinen Trainingsdaten und fügt sie in den Prompt ein.

Dauer: ~10 Sekunden. Verbesserung: oft 10-30%.

Schau dir den Baseline-Score an — nicht schlecht, aber nicht gut genug. Dann drück auf **Optimize** und beobachte, wie der Score steigt!

In [ ]:
btn = run_button("Optimize Multi-hop QA")
out = widgets.Output()

def on_optimize(b):
    with out:
        out.clear_output()
        task = get_task("multihop_qa")
        print(f"⏳ Optimizing {task.name} with {opt_dd.value} on {model_dd.value}...")
        print(f"   This may take 10-60 seconds...\n")

        result = run_optimization("multihop_qa", opt_dd.value, max_eval=8)

        display_improvement(result.baseline_score, result.optimized_score)
        print(f"⏱️  Optimization took {result.elapsed_seconds}s | {result.llm_calls} LLM calls")

        # THE KEY MOMENT: Show what changed
        display_prompt_diff(result.prompt_before, result.prompt_after)

        display_insight("Du vs. Maschine",
            f"Der Optimizer hat {result.optimized_score:.0%} erreicht. "
            "Wie war dein manueller Score? Die Maschine probiert systematisch "
            "tausende Varianten — das ist der Vorteil von automatischer Optimierung.")

btn.on_click(on_optimize)
display(btn, out)

## MIPROv2: Der Heavy-Duty Compiler

**MIPROv2** ist wie `-O3` — er optimiert gleichzeitig die **Instruktionen UND die Beispiele** mit Bayesian Search. Mächtiger, aber langsamer.

Dauer: ~30-60 Sekunden. Verbesserung: oft nochmal besser als BootstrapFewShot.

In [ ]:
compare_btn = run_button("Compare Optimizers")
compare_out = widgets.Output()

def on_compare_opt(b):
    with compare_out:
        compare_out.clear_output()
        task = get_task("ticket_routing")
        print(f"⏳ Comparing optimizers on {task.name}...\n")

        r_bs = run_optimization("ticket_routing", "BootstrapFewShot", max_eval=8)
        print(f"BootstrapFewShot: {r_bs.baseline_score:.0%} → {r_bs.optimized_score:.0%} ({r_bs.elapsed_seconds}s)")

        r_mipro = run_optimization("ticket_routing", "MIPROv2", max_eval=8)
        print(f"MIPROv2:          {r_mipro.baseline_score:.0%} → {r_mipro.optimized_score:.0%} ({r_mipro.elapsed_seconds}s)")

        scores = {
            "BootstrapFewShot": {"baseline": r_bs.baseline_score, "optimized": r_bs.optimized_score},
            "MIPROv2": {"baseline": r_mipro.baseline_score, "optimized": r_mipro.optimized_score},
        }
        fig = bar_comparison("Ticket Routing: Optimizer Comparison", scores)
        fig.show()

compare_btn.on_click(on_compare_opt)
display(compare_btn, compare_out)

In [ ]:
all_tasks = [(t.name, t.id) for t in [get_task(tid) for tid in ["multihop_qa", "ticket_routing", "report_generation"]]]
task_dd = widgets.Dropdown(options=all_tasks, description="Task:")
optimize_btn = run_button("Optimize & Show Diff")
optimize_out = widgets.Output()

def on_any_optimize(b):
    with optimize_out:
        optimize_out.clear_output()
        result = run_optimization(task_dd.value, opt_dd.value, max_eval=8)
        display_improvement(result.baseline_score, result.optimized_score)
        display_prompt_diff(result.prompt_before, result.prompt_after)

optimize_btn.on_click(on_any_optimize)
display(widgets.HBox([task_dd, optimize_btn]), optimize_out)

## 🏆 Vergleich: Du vs. Maschine

Schau dir den Unterschied an:
- **Dein bester manueller Prompt:** Den hast du oben geschrieben und getestet
- **Automatisch optimierter Prompt:** Den hat der Optimizer gefunden

Der optimierte Prompt enthält oft:
- Präzisere Anweisungen (die du vielleicht nicht formuliert hättest)
- Automatisch ausgewählte Few-Shot-Beispiele (die besten aus den Trainingsdaten)
- Instruktionen die spezifisch auf die Fehler zugeschnitten sind

> 💡 **Das ist der Punkt:** Manuelles Prompt-Tuning funktioniert, ist aber langsam und fragil. Automatische Optimierung findet bessere Prompts, schneller, reproduzierbar. **Deine Metrik + Daten = dein Programm. Der Optimizer ist der Compiler.**

## ⏭️ Weiter geht's!

Der Optimizer hat den Prompt verbessert — automatisch, messbar, reproduzierbar. Aber was passiert, wenn du **deine eigenen, echten Daten** benutzt?

Im nächsten Notebook nehmen wir die echten Ticket-Daten aus dem Projekt und zeigen: **Deine Daten sind dein Burggraben.** Ein generisches Modell + deine Domain-Daten + Tuning = etwas, das kein Konkurrent kopieren kann.

👉 **[Weiter zu Notebook: Domain-Tuning →](03_domain_tuning.ipynb)**
